# Day 09: `nn.Module` — The Standard Way to Build Models

**Goal:** Stop writing `w = torch.randn(..., requires_grad=True)` by hand. Start using PyTorch's building blocks like a pro.

### The transition

```python
# DAY 1-7: manual tensor management
w1 = torch.randn(2, 4, requires_grad=True)
b1 = torch.zeros(4, requires_grad=True)
w2 = torch.randn(4, 1, requires_grad=True)
b2 = torch.zeros(1, requires_grad=True)
def forward(x):
    h = torch.relu(x @ w1 + b1)
    return torch.sigmoid(h @ w2 + b2)

# DAY 9 ONWARDS: nn.Module
class XORNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc1 = nn.Linear(2, 4)
        self.fc2 = nn.Linear(4, 1)
    def forward(self, x):
        return self.fc2(torch.relu(self.fc1(x)))
```

Same network. Way cleaner code. Easier to extend.

Today we'll learn:
1. The `nn.Module` template (`__init__` and `forward`)
2. Pre-built layers (`nn.Linear`, `nn.ReLU`, etc.)
3. `nn.Sequential` for quick stacking
4. `nn.ModuleList` for flexible layer collections
5. Inspecting models (parameter counts, named layers)
6. Saving and loading
7. Rebuilding XOR with all our new infrastructure

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt

torch.manual_seed(42)

## 1. The `nn.Module` Template

Every model inherits from `nn.Module` and implements two methods:

```python
class MyModel(nn.Module):
    def __init__(self):           # define your layers here
        super().__init__()         # MUST call this first!
        self.layer1 = nn.Linear(10, 5)
        self.layer2 = nn.Linear(5, 1)
    
    def forward(self, x):          # define how data flows through layers
        x = self.layer1(x)
        x = torch.relu(x)
        x = self.layer2(x)
        return x
```

Then use it:

```python
model = MyModel()
output = model(input)              # NOT model.forward(input)
```

When you call `model(x)`:
1. PyTorch calls `forward(x)` for you
2. It tracks the computation for autograd
3. It returns the output

In [ ]:
# Build your first nn.Module model

class MyFirstModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc1 = nn.Linear(2, 4)   # 2 inputs → 4 outputs
        self.fc2 = nn.Linear(4, 1)   # 4 inputs → 1 output
    
    def forward(self, x):
        x = self.fc1(x)               # apply first layer
        x = torch.relu(x)             # ReLU activation
        x = self.fc2(x)               # apply second layer
        return x

# Create and use it
model = MyFirstModel()
print("Model architecture:")
print(model)

# Test it with some data
x = torch.randn(3, 2)                # 3 samples, 2 features each
print(f"\nInput shape:  {x.shape}")
output = model(x)
print(f"Output shape: {output.shape}")
print(f"Output:\n{output}")

In [ ]:
# What did nn.Linear give us automatically?
# Inspecting parameters

print("Layer 1 (fc1):")
print(f"  Weight shape: {model.fc1.weight.shape}")     # (4, 2) — out × in
print(f"  Bias shape:   {model.fc1.bias.shape}")       # (4,)
print(f"  Weight requires_grad: {model.fc1.weight.requires_grad}")

print("\nLayer 2 (fc2):")
print(f"  Weight shape: {model.fc2.weight.shape}")     # (1, 4)
print(f"  Bias shape:   {model.fc2.bias.shape}")       # (1,)

# All learnable parameters in one place
print("\nAll parameters via model.parameters():")
total = 0
for name, param in model.named_parameters():
    n = param.numel()
    total += n
    print(f"  {name:>15}: shape {tuple(param.shape)} → {n} params")
print(f"\n  Total: {total} parameters")
print("\nNotice: PyTorch automatically created these tensors with requires_grad=True!")
print("No more manual torch.randn(..., requires_grad=True) for every weight.")

## 2. `nn.Sequential` — Quick Stacking for Simple Networks

If your model is just "run these layers in order," skip the class and use `nn.Sequential`:

In [ ]:
# Same network as MyFirstModel, but using Sequential

model_seq = nn.Sequential(
    nn.Linear(2, 4),
    nn.ReLU(),       # ← ReLU is now a "layer" in the stack
    nn.Linear(4, 1),
)

print("Sequential model:")
print(model_seq)

print(f"\nParam count: {sum(p.numel() for p in model_seq.parameters())}")

# Use it the same way
x = torch.randn(3, 2)
out = model_seq(x)
print(f"\nOutput shape: {out.shape}")
print(f"Output: {out}")

print("\nSequential is great when the network is just a chain.")
print("For anything custom (skip connections, conditional logic), use a class.")

## 3. The Common Layers You'll Use

A quick tour of layers you'll see in every PyTorch project:

In [ ]:
# Tour of common layers

# 1. nn.Linear — the workhorse (fully connected layer)
linear = nn.Linear(in_features=10, out_features=5)
x = torch.randn(2, 10)            # batch of 2 samples, 10 features each
print(f"Linear(10, 5):    {x.shape} → {linear(x).shape}")

# 2. nn.ReLU — non-linearity
relu = nn.ReLU()
x = torch.tensor([-2.0, -1.0, 0.0, 1.0, 2.0])
print(f"\nReLU input:  {x}")
print(f"ReLU output: {relu(x)}    (clamps negatives to 0)")

# 3. nn.Sigmoid, nn.Tanh, nn.GELU
print(f"\nSigmoid(0) = {nn.Sigmoid()(torch.tensor(0.0)):.3f}  (squishes to 0-1)")
print(f"Tanh(0)    = {nn.Tanh()(torch.tensor(0.0)):.3f}  (squishes to -1 to 1)")
print(f"GELU(0)    = {nn.GELU()(torch.tensor(0.0)):.3f}  (smooth ReLU — used in transformers)")

# 4. nn.Dropout — randomly zeroes some values during training (regularization)
dropout = nn.Dropout(p=0.5)
x = torch.ones(10)
print(f"\nDropout (in train mode):")
dropout.train()
print(f"  Input:  {x}")
print(f"  Output: {dropout(x)}    (~half are zero!)")
print("  In eval mode, dropout does NOTHING — just passes through")

# 5. nn.LayerNorm — normalizes across features (used in transformers)
ln = nn.LayerNorm(4)
x = torch.tensor([[1.0, 2.0, 3.0, 4.0]])
out = ln(x)
print(f"\nLayerNorm input: {x}")
print(f"LayerNorm output: {out}")
print(f"  → mean ≈ 0, std ≈ 1 across the feature dim")

## 4. Reusable Building Blocks

The real power of `nn.Module` is making your own custom blocks that you can reuse:

In [ ]:
# Define a reusable "MLP block" — Linear → activation → dropout

class MLPBlock(nn.Module):
    """A standard MLP block: Linear + ReLU + Dropout."""
    
    def __init__(self, in_dim, out_dim, dropout=0.1):
        super().__init__()
        self.linear = nn.Linear(in_dim, out_dim)
        self.activation = nn.ReLU()
        self.dropout = nn.Dropout(dropout)
    
    def forward(self, x):
        x = self.linear(x)
        x = self.activation(x)
        x = self.dropout(x)
        return x

# Now build a deep network by stacking these blocks
class DeepMLP(nn.Module):
    def __init__(self, dims=[10, 64, 64, 32, 1], dropout=0.1):
        """
        dims: list of layer sizes, e.g. [10, 64, 64, 32, 1]
              creates: 10 → 64 → 64 → 32 → 1
        """
        super().__init__()
        # nn.ModuleList: like a Python list but PyTorch tracks the layers
        # NEVER use a regular Python list — PyTorch won't see those layers!
        self.blocks = nn.ModuleList([
            MLPBlock(dims[i], dims[i+1], dropout)
            for i in range(len(dims) - 2)         # all but last
        ])
        # Final layer: no activation/dropout (output layer)
        self.final = nn.Linear(dims[-2], dims[-1])
    
    def forward(self, x):
        for block in self.blocks:
            x = block(x)
        return self.final(x)

# Build a deep network with 5 layers
model = DeepMLP(dims=[10, 64, 64, 32, 1])
print(model)
print(f"\nTotal parameters: {sum(p.numel() for p in model.parameters()):,}")

# Test
x = torch.randn(8, 10)
out = model(x)
print(f"\nInput:  {x.shape}")
print(f"Output: {out.shape}")

### Critical: `nn.ModuleList` vs Python list

A common mistake — let's see what happens with each:

In [ ]:
# BAD — using a regular Python list (DON'T DO THIS!)
class BrokenModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.layers = [nn.Linear(10, 5), nn.Linear(5, 1)]   # ✗ Python list
    
    def forward(self, x):
        for layer in self.layers:
            x = layer(x)
        return x

# GOOD — using nn.ModuleList
class WorkingModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.layers = nn.ModuleList([nn.Linear(10, 5), nn.Linear(5, 1)])  # ✓
    
    def forward(self, x):
        for layer in self.layers:
            x = layer(x)
        return x

broken = BrokenModel()
working = WorkingModel()

print("BrokenModel.parameters():")
broken_params = list(broken.parameters())
print(f"  Found {len(broken_params)} parameters")
print(f"  Total values: {sum(p.numel() for p in broken_params)}")
print("  ⚠️  PyTorch can't see the layers! Optimizer won't update them!")

print("\nWorkingModel.parameters():")
working_params = list(working.parameters())
print(f"  Found {len(working_params)} parameters")
print(f"  Total values: {sum(p.numel() for p in working_params)}")
print("  ✓  PyTorch tracks all layers correctly")

print("\nLesson: ALWAYS use nn.ModuleList (or nn.Sequential) — NEVER a Python list.")

## 5. Built-in Module Methods

`nn.Module` gives you a bunch of useful methods for free:

In [ ]:
model = DeepMLP(dims=[10, 32, 16, 1])

# 1. Move to GPU/CPU
device = 'mps' if torch.backends.mps.is_available() else 'cpu'
model = model.to(device)
print(f"Model on device: {next(model.parameters()).device}")

# 2. Train mode vs Eval mode (important for dropout, batchnorm)
model.train()           # enables dropout etc.
print(f"\nIn train mode: dropout active")

model.eval()            # disables dropout
print(f"In eval mode: dropout disabled")

# 3. state_dict — all parameters as a dictionary (for saving)
state = model.state_dict()
print(f"\nstate_dict keys (first 5):")
for key in list(state.keys())[:5]:
    print(f"  {key}: shape {state[key].shape}")

# 4. Save to disk
torch.save(model.state_dict(), '/tmp/day09_model.pth')
print("\nSaved to /tmp/day09_model.pth")

# 5. Load from disk
new_model = DeepMLP(dims=[10, 32, 16, 1]).to(device)
new_model.load_state_dict(torch.load('/tmp/day09_model.pth'))
print("Loaded into new_model")

# 6. Verify they're identical
x = torch.randn(2, 10).to(device)
out1 = model(x)
out2 = new_model(x)
match = torch.allclose(out1, out2)
print(f"\nOriginal and loaded model produce same output: {match}")

## 6. Putting It All Together — XOR with the New Stack

Let's rebuild XOR using EVERYTHING from Days 8 and 9:
- `nn.Module` model
- Adam optimizer + cosine LR scheduler
- `BCEWithLogitsLoss`
- Train/val tracking
- Best model saving

In [ ]:
# XOR — using nn.Module + everything from Day 08

# 1. Define the model
class XORNet(nn.Module):
    def __init__(self, hidden=8):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(2, hidden),
            nn.ReLU(),
            nn.Linear(hidden, hidden),
            nn.ReLU(),
            nn.Linear(hidden, 1),     # logits — no sigmoid here
        )
    
    def forward(self, x):
        return self.net(x)

# 2. Setup
torch.manual_seed(42)
device = 'mps' if torch.backends.mps.is_available() else 'cpu'
model = XORNet(hidden=8).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=0.05)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=200)
loss_fn = nn.BCEWithLogitsLoss()

# 3. XOR data
X = torch.tensor([[0.0, 0.0], [0.0, 1.0], [1.0, 0.0], [1.0, 1.0]]).to(device)
y = torch.tensor([[0.0], [1.0], [1.0], [0.0]]).to(device)

# 4. Training loop with best model saving
losses = []
lrs = []
best_loss = float('inf')

for epoch in range(200):
    lrs.append(optimizer.param_groups[0]['lr'])
    
    model.train()
    logits = model(X)
    loss = loss_fn(logits, y)
    
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    scheduler.step()
    
    losses.append(loss.item())
    
    # Save best model
    if loss.item() < best_loss:
        best_loss = loss.item()
        torch.save(model.state_dict(), '/tmp/best_xor.pth')

# 5. Load the best model and evaluate
model.load_state_dict(torch.load('/tmp/best_xor.pth'))
model.eval()
with torch.no_grad():
    logits = model(X)
    probs = torch.sigmoid(logits)

# 6. Print results
print("XOR predictions (using nn.Module + Day 08 infrastructure):\n")
print(f"{'Input':>10} | {'Target':>6} | {'Predicted':>9} | {'Correct?':>8}")
print("-" * 50)
for i in range(4):
    x1, x2 = X[i, 0].item(), X[i, 1].item()
    t = y[i, 0].item()
    p = probs[i, 0].item()
    correct = "✓" if round(p) == t else "✗"
    print(f"  ({x1:.0f}, {x2:.0f})   | {t:>6.0f} | {p:>9.3f} | {correct:>8}")

print(f"\nBest loss: {best_loss:.6f}")
print(f"Total parameters: {sum(p.numel() for p in model.parameters())}")

# 7. Plot
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 4))
ax1.plot(losses, 'b-', linewidth=2)
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Loss')
ax1.set_title('XOR loss curve')
ax1.set_yscale('log')
ax1.grid(True, alpha=0.3)

ax2.plot(lrs, 'm-', linewidth=2)
ax2.set_xlabel('Epoch')
ax2.set_ylabel('Learning rate')
ax2.set_title('Cosine LR schedule')
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

---

## Exercises

1. **Add dropout:** Modify XORNet to include `nn.Dropout(0.2)` between layers. Does training still work?

2. **Try GELU:** Replace `nn.ReLU()` with `nn.GELU()` (the activation used in transformers). Compare loss curves.

3. **Inspect parameters:** For your XORNet, print every parameter's name and shape using `named_parameters()`. Which one has the most weights?

4. **Build a regression model:** Define a network with `nn.Sequential` to fit a sine curve. Use 1 input, hidden dims [32, 64, 32], 1 output.

5. **Two ways, same model:** Build the same XOR network using (a) `nn.Sequential` and (b) a custom class with explicit `__init__`/`forward`. Which feels cleaner?

---

## Key Takeaways

### The two methods every nn.Module needs

```python
class MyModel(nn.Module):
    def __init__(self):           # define WHAT layers exist
        super().__init__()         # ALWAYS call this first!
        self.fc1 = nn.Linear(...)
        self.fc2 = nn.Linear(...)
    
    def forward(self, x):          # define HOW data flows
        return self.fc2(F.relu(self.fc1(x)))
```

### Storing layers — use the right container

| Container | When to use |
|-----------|-------------|
| `nn.Sequential(...)` | Simple chain: layer → layer → layer |
| `nn.ModuleList([...])` | List of layers needing custom forward logic |
| Python list (`[...]`) | **NEVER** — PyTorch won't track the layers |

### Common layers cheat sheet

```python
nn.Linear(in, out)         # fully connected
nn.ReLU() / nn.GELU()      # activation
nn.Dropout(p)               # regularization (p% chance to zero each value)
nn.LayerNorm(dim)           # normalize across feature dim
nn.Embedding(vocab, dim)    # token ID → vector (we'll use this for LLMs!)
```

### Free methods you get from nn.Module

```python
model.parameters()           # all learnable tensors
model.named_parameters()     # tensors with names
model.to(device)             # move everything to GPU
model.train() / model.eval() # toggle dropout/batchnorm mode
model.state_dict()           # all weights as a dict
model.load_state_dict(...)   # restore weights from a dict
```

### The standard PyTorch model template

```python
class MyModel(nn.Module):
    def __init__(self, ...):
        super().__init__()
        # define layers
    
    def forward(self, x):
        # define computation
        return x

model = MyModel(...)
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
loss_fn = nn.SomeLoss()

for epoch in range(epochs):
    for x, y in dataloader:
        loss = loss_fn(model(x), y)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
```

This is the template for every model from here on out.

**Tomorrow (Day 10):** We finally tackle TEXT — building a sentiment classifier on movie reviews. Time to put all this to work on real data!